# RAG on Azure — Day 11: Sentence-Window Retrieval 🪟

**Use case:** Day 10 gave the LLM a *full section* of context. Sometimes that's overkill — the answer fits in a few consecutive sentences, and the rest of the section is noise that dilutes the signal (and burns tokens). Day 11 zooms in: index individual **sentences**, then expand each match to a small **±N window** of neighbors.

**What's new vs Day 10:**
- **Sentence-level indexing.** Each record is a single sentence, with a pre-computed `window` text containing the sentence plus its N neighbors on each side.
- The retrieval call returns the window directly (the dedup step keeps overlapping windows out of the same answer).
- A head-to-head demo against Day 10's parent-doc retrieval, showing where each technique wins.

**Done when:** you can articulate, with a specific example, when sentence-window beats parent-doc and when parent-doc beats sentence-window.

**Data:** reuses `data-day10/` — the 3 enterprise policy PDFs. No new files.

**Prereqs:** Day 10's index `rag-policies-day10` should still exist (we read from it in the comparison cells). If not, re-run Day 10 first.

## 0. Install dependencies

In [ ]:
%pip install -q -r ../requirements.txt

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from common import load_config, get_openai_client, get_search_client, get_search_index_client
from common import load_pdfs, extract_sections
from common import split_sentences, make_sentence_windows
from common import embed_texts, retrieve_parents, retrieve_windows
from common import answer

config = load_config()
openai = get_openai_client(config)
index_client = get_search_index_client(config)
INDEX_NAME       = "rag-policies-day11"      # sentence-level index built here
INDEX_NAME_DAY10 = "rag-policies-day10"      # parent-doc index from Day 10, for comparison
print("Setup OK. Day 11 index:", INDEX_NAME)

## 2. Load PDFs and split into sentences with windows

For each document, we extract sections (so a window doesn't cross a section boundary), split each section into sentences, and pre-compute a `±N` window for every sentence. Pre-computing the window at indexing time is the standard pattern — it costs storage but makes retrieval a single round-trip.

In [ ]:
from pathlib import Path
import re

DATA_DIR = Path("../data-day10")
WINDOW_SIZE = 2                    # +/- N sentences around each match

print("Looking in:", DATA_DIR.resolve())
raw_docs = load_pdfs(DATA_DIR)
assert raw_docs, f"No PDFs in {DATA_DIR.resolve()}"

records = []
for d in raw_docs:
    for section_title, section_body in (extract_sections(d["text"]) or [("Body", d["text"])]):
        sentences = split_sentences(section_body)
        windows = make_sentence_windows(sentences, window_size=WINDOW_SIZE)
        safe_src = re.sub(r"[^A-Za-z0-9_-]", "_", d["source"])
        safe_sec = re.sub(r"[^A-Za-z0-9_-]", "_", section_title)[:30]
        for w in windows:
            records.append({
                "id":             f"{safe_src}-{safe_sec}-s{w['sentence_index']}",
                "sentence":       w["sentence"],         # embedded
                "window":         w["window"],           # sent to LLM at query time
                "sentence_index": w["sentence_index"],
                "source":         d["source"],
                "section":        section_title,
            })

print(f"\n{len(records)} sentences across {len(raw_docs)} PDFs.")
print(f"\nExample sentence + its plus/minus {WINDOW_SIZE} window:")
ex = records[3]
print(f"  matched sentence ({len(ex['sentence'])} chars):")
print(f"    \"{ex['sentence']}\"")
print(f"  expanded window ({len(ex['window'])} chars):")
print(f"    \"{ex['window']}\"")

## 3. Embed (one vector per sentence)

Embedding individual sentences gives the most precise vector you can get — small, focused units of meaning.

In [ ]:
vectors = embed_texts(openai, [r["sentence"] for r in records], config["embedding_deployment"])
EMBED_DIM = len(vectors[0])
print(f"Embedded {len(vectors)} sentences. Dim: {EMBED_DIM}")

## 4. Create the sentence-level index

The new field shape: `sentence` is embedded; `window` is the pre-expanded text we send to the LLM at query time. `sentence_index` is filterable (useful for window-dedup or filtering by position).

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="sentence", type=SearchFieldDataType.String),    # embedded
    SearchableField(name="window",   type=SearchFieldDataType.String),    # stored, sent to LLM
    SimpleField(name="sentence_index", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SimpleField(name="source",  type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="section", type=SearchFieldDataType.String, filterable=True),
    SearchField(
        name="contentVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBED_DIM,
        vector_search_profile_name="vprofile",
    ),
]
vs = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vprofile", algorithm_configuration_name="hnsw")],
)

try:
    index_client.delete_index(INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'.")
except Exception as e:
    print("No existing index (ok):", type(e).__name__)

index_client.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vs))
print(f"Index '{INDEX_NAME}' created with sentence + window fields.")

## 5. Upload

In [ ]:
search = get_search_client(config, INDEX_NAME)
docs = [{**r, "contentVector": v} for r, v in zip(records, vectors)]
for i in range(0, len(docs), 100):
    search.upload_documents(documents=docs[i:i+100])
print(f"Uploaded {len(docs)} sentence records.")

## 6. Compare side by side — parent-doc vs sentence-window

Same question, two retrieval strategies, two different "context shapes" reaching the LLM:
- **Parent-doc** (Day 10): the full section the match belongs to (~1500 chars per parent)
- **Sentence-window** (Day 11): the matched sentence + ±N neighbors (~400 chars per window)

The cells below print both so you can see context size and answer quality at the same time.

In [ ]:
SELECT_DAY10 = ["content", "parent_content", "parent_id", "source", "section"]
SELECT_DAY11 = ["sentence", "window", "sentence_index", "source", "section"]

search_day10 = get_search_client(config, INDEX_NAME_DAY10)

def compare(question, k=3):
    print("=" * 80)
    print(f"Q: {question}\n")

    parents = retrieve_parents(
        search_day10, openai, question, config["embedding_deployment"],
        k_parents=k, select=SELECT_DAY10,
    )
    windows = retrieve_windows(
        search, openai, question, config["embedding_deployment"],
        k_windows=k, k_sentences=12, select=SELECT_DAY11,
    )

    print(f"-- PARENT-DOC (Day 10) — {sum(len(p['content']) for p in parents)} chars total --")
    for p in parents:
        print(f"   {p['source']} / {p['section']}  ({len(p['content'])} chars)")
    a_parent = answer(openai, config["chat_deployment"], question, parents, cite=True)
    print(f"\n   Answer: {a_parent}\n")

    print(f"-- SENTENCE-WINDOW (Day 11) — {sum(len(w['content']) for w in windows)} chars total --")
    for w in windows:
        print(f"   {w['source']} / {w['section']}  ({len(w['content'])} chars)")
        print(f"      matched: \"{w['matched_sentence'][:90]}{'...' if len(w['matched_sentence'])>90 else ''}\"")
    a_window = answer(openai, config["chat_deployment"], question, windows, cite=True)
    print(f"\n   Answer: {a_window}\n")

## 7. Demo A — focused factual question

The answer is *one specific sentence* with one neighbor for full context. Parent-doc sends ~1500 chars of mostly-unrelated section text. Sentence-window sends ~400 chars — just enough.

In [ ]:
compare("How quickly must I report a damaged item on a standard order?")

## 8. Demo B — the "exception in the next paragraph" case

This is where parent-doc usually wins. The exception lives 2–3 sentences past the matched sentence — which may or may not fit in a ±2 window. Try increasing `WINDOW_SIZE` if Day 11 misses it; that's the trade-off in action.

In [ ]:
compare("Can I return a custom-made item that arrived damaged?")

## 9. Demo C — short, factual answer

Best case for sentence-window: the answer is literally one sentence. Parent-doc sends ~10× more text for the same payoff.

In [ ]:
compare("When are credit card refunds typically posted?")

## 10. Context-size comparison

A simple table showing how many characters each strategy sends to the LLM. Across many queries this difference compounds into real cost.

In [ ]:
questions = [
    "How quickly must I report a damaged item on a standard order?",
    "Can I return a custom-made item that arrived damaged?",
    "When are credit card refunds typically posted?",
    "How long does it take to process my data deletion request?",
    "If a vendor has a negative finding, are they automatically rejected?",
]

print(f"{'Question':<60} {'Parent-doc':>11} {'Window':>9}  {'Saved':>7}")
print("-" * 96)
for q in questions:
    parents = retrieve_parents(search_day10, openai, q, config["embedding_deployment"], k_parents=3, select=SELECT_DAY10)
    windows = retrieve_windows(search,        openai, q, config["embedding_deployment"], k_windows=3, k_sentences=12, select=SELECT_DAY11)
    p_chars = sum(len(p["content"]) for p in parents)
    w_chars = sum(len(w["content"]) for w in windows)
    saved = (p_chars - w_chars) / max(p_chars, 1) * 100
    print(f"{q[:58]:<60} {p_chars:>11} {w_chars:>9}  {saved:>6.0f}%")

## ✅ Day 11 checklist

- [ ] Each indexed record is a single sentence with a pre-computed `window` field
- [ ] `retrieve_windows()` returns the window text in `content` (drops into `answer()` unchanged)
- [ ] On Demo A (focused factual), sentence-window sends much less context than parent-doc — for the same correct answer
- [ ] On Demo B (exception across paragraphs), parent-doc gives the more complete answer
- [ ] You can articulate one concrete question type for each technique

### What you learned today
- **Two valid context strategies, different shapes.** Parent-doc is wide and structured; sentence-window is narrow and surgical. Neither is universally better.
- **Pick by question type.** Factual lookups → sentence-window. Nuanced policy questions with cross-paragraph exceptions → parent-doc. Production systems sometimes run both and pick per-query.
- **Token cost matters at scale.** A large context-size reduction is a large cost reduction on the generation call. For high-volume chatbots that adds up to real money.

### Next up — Day 12: contextual retrieval
A different angle on the same problem. Instead of *expanding* context at query time, we *enrich* each chunk at indexing time with a summary of the document it came from — so retrieval gets smarter automatically, without making the context bigger.